# Does the Algorithm Hide Europe?

## Final Project Roadmap and Research-Question Answers


**Team:** Max Priessnitz & Nico [surname]

**Course:** Data Science and Artificial Intelligence II: Data and Algorithmic Governance

**Purpose:** This short notebook is the entry point for the final submission. It explains the reading order, the architecture, and the main answers. The full evidence is in `04_final_research_story_executed.ipynb`.


The storyline is deliberately simple: streaming platforms can satisfy catalogue-level diversity while ranking systems still decide what users actually see. We therefore audit cultural prominence at the Top-K ranking layer, not only catalogue availability.

## Reading order


1. `00_project_roadmap_and_final_answers.ipynb` — this entry notebook.

2. `01_data_foundation_movies_db.ipynb` — MovieLens/M3L/Wikidata data foundation.

3. `02_model_pipeline_and_user_fold_robustness.ipynb` — recommender models, metrics and user-fold robustness.

4. `03_feedback_loop_and_mitigation.ipynb` — Schedl-inspired feedback-loop stress test and mitigation logic.

5. `04_final_research_story_executed.ipynb` — full final research notebook with executed outputs, interpretations and limitations.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
TABLES = ROOT / "cultural_prominence_audit" / "outputs" / "final_notebook_tables"
ASSETS = ROOT / "cultural_prominence_audit" / "outputs" / "final_submission_assets"

answers = pd.read_csv(TABLES / "final_answer_table.csv")
answers[["Question", "Short answer", "Confidence", "Main caveat"]].head(18)

,Question,Short answer,Confidence,Main caveat
0,Does the algorithm hide Europe?,Model-dependent. Worst Europe PACPG@20 is Popu...,moderate,Offline sampled audit with proxy metadata.
1,Is catalogue diversity enough?,No. The visibility funnel shows catalogue shar...,strong,Catalogue and metadata coverage still matter.
2,Which Europe gets recommended?,Visibility concentrates unevenly: strongest re...,moderate,Low-support regions are not overclaimed.
3,Does the recommender show local Europe or glob...,Platform-compatible Europe has mean Exposure@2...,exploratory/moderate,DNA scores are transparent proxies; TMDb provi...
4,Which European countries are most visible?,United Kingdom has the highest mean Exposure@2...,moderate,Fractional/binary country counting can differ.
5,Which European countries are least visible?,France has the lowest mean PACPG@20 among supp...,moderate,Low support limits claims.
6,Which languages are most visible?,English has the highest mean language Exposure...,moderate,Original-language metadata is a proxy.
7,Is non-English content underexposed?,Non-English visibility is model-dependent; the...,moderate,Multilingual films complicate binary labels.
8,Are Spanish films visible?,Best Spain-origin exposure model is MPNet-cont...,moderate,Support and metadata coverage limit fine claims.
9,Are Spanish-language films the same as Spain-o...,"No. The data show 227 films with both labels, ...",strong,Depends on Wikidata language metadata.


## What we found, in plain English


- The answer is **model-dependent**, not a single yes/no slogan. Popularity produces the strongest Europe underexposure, while CLIP-image-content shows the highest Europe exposure.

- The best ranking-utility model is **SVD**, but the best cultural-visibility model is not necessarily the same model.

- **English-language content dominates** Top-20 exposure. MPNet-content is the strongest model for non-English exposure in this bounded run.

- European visibility is uneven. France, Spain and Germany show the most discussable support-passing country-level gaps; several smaller countries are near-invisible and must be interpreted with support thresholds.

- The stricter **local Europe** proxy is far less visible than globally compatible Europe. This is why we added the Visibility DNA extension.

- Re-ranking can increase prominence, but the frontier shows a real utility cost. That trade-off is the governance result, not a hidden moral adjustment.

In [2]:
model_summary = pd.read_csv(TABLES / "aggregate_visibility_metrics.csv")
model_summary[["Model", "NDCG@20", "Recall@20", "Coverage@20", "Europe wide Exposure@20", "Non-English Exposure@20", "Europe wide PACPG@20"]]

,Model,NDCG@20,Recall@20,Coverage@20,Europe wide Exposure@20,Non-English Exposure@20,Europe wide PACPG@20
0,Popularity,0.118636,0.129146,0.040895,0.097407,0.000087,-0.103187
1,ItemKNN,0.162323,0.200195,0.160138,0.148670,0.002903,-0.051924
2,SVD,0.181420,0.233202,0.352561,0.187734,0.021240,-0.012860
3,MPNet-content,0.019410,0.023218,0.176496,0.181217,0.087510,-0.019377
4,CLIP-image-content,0.014402,0.013126,0.102884,0.327980,0.009676,0.127386
5,Hybrid,0.166821,0.210883,0.276797,0.182289,0.013167,-0.018305


In [3]:
country_scorecard = pd.read_csv(ASSETS / "country_problem_scorecard.csv")
country_scorecard[["group_name", "mean_exposure_pct", "gap_pp", "top_genres", "problem_type"]].head(12)

,group_name,mean_exposure_pct,gap_pp,top_genres,problem_type
0,France,2.967430,-1.721219,Drama | Comedy | Romance,underexposed relative to target
1,Spain,0.141078,-0.490734,Drama | Comedy | Thriller,near target or low-support signal
2,Germany,2.821971,-0.406706,Drama | Comedy | Thriller,near target or low-support signal
3,Poland,0.008286,-0.326983,Drama | Comedy | Mystery,near-invisible despite target signal
4,Netherlands,0.004780,-0.234835,Drama | Comedy | Thriller,near-invisible despite target signal
5,Switzerland,0.034868,-0.200972,Drama | Romance | Comedy,near-invisible despite target signal
6,Sweden,0.004132,-0.172260,Drama | Comedy | Thriller,near-invisible despite target signal
7,Austria,0.021537,-0.169039,Drama | Comedy | Documentary,near-invisible despite target signal
8,Norway,0.025512,-0.085605,Drama | Action | Comedy,near-invisible despite target signal
9,Denmark,0.000000,-0.081278,Drama | Thriller | Comedy,near target or low-support signal


In [4]:
company_caveats = pd.read_csv(ASSETS / "production_company_caveat_summary.csv")
company_caveats

,segment,movies,share_of_catalogue
0,European-origin films with US production-compa...,1155,0.042342
1,Non-US-origin films with US production-company...,354,0.012977
2,Non-English films with US production-company/H...,98,0.003593
3,Films missing production-company country/HQ me...,17508,0.641836


## How to use this submission


For grading, start with this roadmap notebook and then open the final research-story notebook for the detailed evidence. The HTML exports contain saved output state; the `.ipynb` files document the reproducible analysis path when the required local raw datasets are available. Raw MovieLens and M3L files are intentionally not redistributed.